In [ ]:
import os
import sys
from pathlib import Path

library_path = os.path.abspath('../src')
if library_path not in sys.path:
    sys.path.append(library_path)
library_path = Path(library_path)
library_path

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.robust.robust_linear_model import RLM

In [ ]:
# load data
DATA_PATH = library_path.parent / "data"
PLOTS_PATH = library_path.parent / "plots"

df = pd.read_csv(f"{DATA_PATH}/all_data.csv", sep="\t")

In [ ]:
cols_to_use = ["Sex", "Age", "Tumor", "CC", "PreOP CTx", "Thermoablation", "sPCI", "pPCI"]
df = df[cols_to_use].copy()

In [ ]:
# data wrangling
df['Sex'] = df['Sex']-1
df['PreOP CTx'] = df['PreOP CTx'].apply(lambda x: 1 if x >= 1 else x)

keep_tumor = [1, 5, 4, 6, 7, 3, 2]  # remove 8 if you decide to drop it

# Filter rows
df = df[df["Tumor"].isin(keep_tumor)].copy()
df['Tumor'] = df['Tumor'].apply(lambda x: f"type_{x}")

df = df[(df['CC']==0) | (df['CC']==1)].reset_index(drop=True)  # keep only CC0 and CC1

In [ ]:
# --- 1. Rename columns ---
df = df.rename(columns={
    "PreOP CTx" : "PreOP_CTx",
})

In [ ]:
# --- 5. One-hot encode Tumor (most frequent as reference) ---
# Tumor "1" (n=122) is the natural reference category
df = pd.get_dummies(df, columns=["Tumor"], drop_first=False, dtype=int)
df = df.drop(columns=["Tumor_type_1"], inplace=False)  # explicitly set Tumor_1 as reference

# --- 6. Create outcome variables ---
df["raw_diff"] = df["sPCI"] - df["pPCI"]
df["abs_diff"] = np.abs(df["raw_diff"])


In [ ]:
feature_cols = [
    'Sex', 'Age', 'CC', 'PreOP_CTx', 'Thermoablation', 'Tumor_type_2', 'Tumor_type_3', 'Tumor_type_4', 'Tumor_type_5',
       'Tumor_type_6', 'Tumor_type_7'
]

X     = df[feature_cols]
y_raw = df["raw_diff"]
y_abs = df["abs_diff"]

print(f"Final feature matrix: {X.shape}")

In [ ]:
X = sm.add_constant(X)
model_rlm = RLM(y_raw, X).fit()
print(model_rlm.summary())

In [ ]:
plt.scatter(model_rlm.fittedvalues, model_rlm.resid)
plt.axhline(0, linestyle='--')

In [ ]:
w = model_rlm.weights
y_wm = np.average(y_raw, weights=w)
ss_res = np.sum(w * model_rlm.resid**2)
ss_tot = np.sum(w * (y_raw - y_wm)**2)
weighted_r2 = 1 - ss_res / ss_tot
print(f"Weighted R²: {weighted_r2:.4f}")

In [ ]:
from scipy.stats import median_abs_deviation

print(f"Residual MAD:  {median_abs_deviation(model_rlm.resid):.4f}")
print(f"Residual RMSE: {np.sqrt(np.mean(model_rlm.resid**2)):.4f}")
print(f"Outcome MAD:   {median_abs_deviation(y_raw):.4f}")
print(f"Outcome RMSE:  {np.sqrt(np.mean((y_raw - y_raw.mean())**2)):.4f}")

## Model Diagnostics

### 1. IRLS Weight Distribution

In [ ]:
from scipy.stats import median_abs_deviation as mad

resid   = model_rlm.resid
fitted  = model_rlm.fittedvalues
weights = model_rlm.weights
scale   = model_rlm.scale
n       = len(y_raw)

huber_thresh = 1.345 * mad(resid)

print(f"N total: {n}")
print(f"w = 1.0  (not downweighted) : {(weights == 1.0).sum():>4}  ({100*(weights==1.0).mean():.1f}%)")
print(f"0.9 <= w < 1.0              : {((weights>=0.9)&(weights<1.0)).sum():>4}")
print(f"0.5 <= w < 0.9              : {((weights>=0.5)&(weights<0.9)).sum():>4}")
print(f"0.2 <= w < 0.5              : {((weights>=0.2)&(weights<0.5)).sum():>4}")
print(f"w < 0.2                     : {(weights<0.2).sum():>4}")
print(f"\nHuber threshold (1.345 * MAD): {huber_thresh:.3f}")
print(f"Residuals beyond threshold   : {(np.abs(resid) > huber_thresh).sum()}")

### 2. Standardised Residuals

In [ ]:
std_resid = resid / scale

print(f"Scale (MAD): {scale:.4f}")
print(f"|std_resid| > 2 : {(np.abs(std_resid) > 2).sum():>4}  ({100*(np.abs(std_resid)>2).mean():.1f}%)")
print(f"|std_resid| > 3 : {(np.abs(std_resid) > 3).sum():>4}  ({100*(np.abs(std_resid)>3).mean():.1f}%)")
print(f"|std_resid| > 4 : {(np.abs(std_resid) > 4).sum():>4}")
print(f"Max |std_resid| : {np.abs(std_resid).max():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(fitted, std_resid, alpha=0.5, color="steelblue")
axes[0].axhline(0,  linestyle="--", color="black")
axes[0].axhline(2,  linestyle=":",  color="orange", label="±2")
axes[0].axhline(-2, linestyle=":",  color="orange")
axes[0].axhline(3,  linestyle=":",  color="red", label="±3")
axes[0].axhline(-3, linestyle=":",  color="red")
axes[0].set_xlabel("Fitted values")
axes[0].set_ylabel("Standardised residuals")
axes[0].set_title("Standardised residuals vs Fitted")
axes[0].legend()

axes[1].hist(std_resid, bins=30, color="steelblue", edgecolor="white")
axes[1].set_xlabel("Standardised residuals")
axes[1].set_title("Distribution of standardised residuals")

plt.tight_layout()
plt.show()

### 3. Normality of Residuals

In [ ]:
from scipy import stats

sw = stats.shapiro(resid)
print(f"Shapiro-Wilk: W={sw.statistic:.4f}, p={sw.pvalue:.4f}")
print(f"Skewness    : {stats.skew(resid):.4f}")
print(f"Kurtosis    : {stats.kurtosis(resid):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(resid, bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Residuals")
axes[0].set_title(f"Residuals distribution\nShapiro-Wilk: W={sw.statistic:.3f}, p={sw.pvalue:.4f}")

sm.qqplot(resid, line="45", ax=axes[1], alpha=0.5)
axes[1].set_title("Q-Q plot of residuals")

plt.tight_layout()
plt.show()

### 4. Autocorrelation (Durbin-Watson)

In [ ]:
dw = np.sum(np.diff(resid.values)**2) / np.sum(resid.values**2)
print(f"Durbin-Watson: {dw:.4f}  (2 = no autocorrelation, <2 = positive, >2 = negative)")

### 5. Residuals vs Predictors (Spearman correlations)

In [ ]:
print(f"{'Predictor':<22} {'rho':>6}  {'p':>7}  {'sig'}")
print("-" * 45)
for col in X.columns:
    r, p = stats.spearmanr(X[col], resid)
    sig = "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else ""))
    print(f"  {col:<20} {r:+.3f}  {p:.4f}  {sig}")

r_fit, p_fit = stats.spearmanr(fitted, resid)
print(f"\n  {'fitted (non-linearity)':<20} {r_fit:+.3f}  {p_fit:.4f}")

### 6. Leverage and Cook's Distance

In [ ]:
X_c = sm.add_constant(X)

# Leverage (hat matrix diagonal)
H_mat = X_c.values @ np.linalg.pinv(X_c.values.T @ X_c.values) @ X_c.values.T
h = np.diag(H_mat)
high_lev_thresh = 2 * X_c.shape[1] / n

print(f"High-leverage threshold (2p/n): {high_lev_thresh:.4f}")
print(f"High-leverage observations    : {(h > high_lev_thresh).sum()} ({100*(h>high_lev_thresh).mean():.1f}%)")
print(f"Max leverage                  : {h.max():.4f}")

# Cook's distance (via OLS as influence proxy)
ols_model = sm.OLS(y_raw, X_c).fit()
infl      = ols_model.get_influence()
cooks_d   = infl.cooks_distance[0]

print(f"\nCook's D > 4/n ({4/n:.4f})    : {(cooks_d > 4/n).sum()} observations")
print(f"Cook's D > 1                  : {(cooks_d > 1).sum()} observations")
print(f"Max Cook's D                  : {cooks_d.max():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].stem(np.arange(n), h, markerfmt="C0.", linefmt="C0-", basefmt="k-")
axes[0].axhline(high_lev_thresh, color="red", linestyle="--", label=f"2p/n = {high_lev_thresh:.3f}")
axes[0].set_xlabel("Observation index")
axes[0].set_ylabel("Leverage")
axes[0].set_title("Leverage")
axes[0].legend()

axes[1].stem(np.arange(n), cooks_d, markerfmt="C1.", linefmt="C1-", basefmt="k-")
axes[1].axhline(4/n, color="red", linestyle="--", label=f"4/n = {4/n:.3f}")
axes[1].set_xlabel("Observation index")
axes[1].set_ylabel("Cook's D")
axes[1].set_title("Cook's Distance")
axes[1].legend()

plt.tight_layout()
plt.show()

### 7. Sensitivity to M-estimator Norm

In [ ]:
import statsmodels.robust.norms as rnorms

norms_to_test = [
    ("HuberT (default)", rnorms.HuberT()),
    ("Bisquare (Tukey)", rnorms.TukeyBiweight()),
    ("AndrewWave",       rnorms.AndrewWave()),
]

rows = []
for name, norm in norms_to_test:
    m = RLM(y_raw, X_c, M=norm).fit(cov="H3")
    row = {"norm": name, "scale": m.scale}
    for col, coef, pval in zip(X_c.columns, m.params, m.pvalues):
        row[f"{col}_coef"] = round(coef, 3)
        row[f"{col}_p"]    = round(pval, 4)
    rows.append(row)

# Summary: coefficients and significance for each norm
print(f"{'Predictor':<22}", end="")
for r in rows:
    print(f"  {r['norm'][:18]:<20}", end="")
print()
print("-" * (22 + 22 * len(rows)))

for col in X_c.columns:
    print(f"{col:<22}", end="")
    for r in rows:
        coef = r[f"{col}_coef"]
        pval = r[f"{col}_p"]
        sig  = "***" if pval < 0.001 else ("**" if pval < 0.01 else ("*" if pval < 0.05 else ""))
        print(f"  {coef:+.3f} (p={pval:.3f}) {sig:<4}", end="")
    print()

print(f"\n{'Scale (MAD)':<22}", end="")
for r in rows:
    print(f"  {r['scale']:.4f}{' '*16}", end="")
print()

### 8. Bootstrap Confidence Intervals

In [ ]:
np.random.seed(42)
n_boot = 2000
boot_params = np.zeros((n_boot, X_c.shape[1]))

for i in range(n_boot):
    idx = np.random.choice(n, size=n, replace=True)
    m_b = RLM(y_raw.iloc[idx], X_c.iloc[idx]).fit(cov="H3")
    boot_params[i] = m_b.params

ci_lo   = np.percentile(boot_params, 2.5,  axis=0)
ci_hi   = np.percentile(boot_params, 97.5, axis=0)
boot_se = boot_params.std(axis=0)

print(f"{'Predictor':<22} {'coef':>7}  {'asym_SE':>8}  {'boot_SE':>8}  {'95% CI (bootstrap)'}")
print("-" * 72)
for pname, coef, ase, bse, lo, hi in zip(
        X_c.columns, model_rlm.params, model_rlm.bse, boot_se, ci_lo, ci_hi):
    sig = "*" if lo > 0 or hi < 0 else ""
    print(f"{pname:<22} {coef:+7.3f}  {ase:8.3f}  {bse:8.3f}  [{lo:+.3f}, {hi:+.3f}] {sig}")